# Practice Lab: GraphRAG & Knowledge Graphs (Lesson 8.7)

Module 8 · 8 exercises · KG construction + traversal + hybrid routing + evaluation

Exercises 1, 6, 7 run locally (pure Python).


# Lesson 8.7: GraphRAG & Knowledge Graphs

8 exercises: 2E + 3M + 3C

Exercises 1, 6, 7 run **locally** (pure Python). Ex 2, 3, 4, 5, 8 are architecture/design.


In [ ]:
import numpy as np
import hashlib
from collections import defaultdict


---
## Exercise 1 (Easy): Knowledge Graph From Scratch


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
from collections import defaultdict

class SimpleKG:
    def __init__(self):
        self.triples = []; self.adj = defaultdict(list)
    def add(self, s, r, o):
        t = (s, r, o); self.triples.append(t)
        self.adj[s.lower()].append(t); self.adj[o.lower()].append(t)
    def neighbors(self, entity, hops=1):
        visited, result, queue = set(), [], [(entity.lower(), 0)]
        while queue:
            node, depth = queue.pop(0)
            if node in visited or depth > hops: continue
            visited.add(node)
            for t in self.adj.get(node, []):
                result.append(t)
                for e in [t[0].lower(), t[2].lower()]:
                    if e not in visited: queue.append((e, depth+1))
        return result
    def query(self, q):
        qw = set(q.lower().split())
        entities = [e for e in self.adj if any(w in e for w in qw)][:3]
        triples = []
        for e in entities: triples.extend(self.neighbors(e, hops=2))
        unique = list(set(triples))
        if unique:
            t = unique[0]
            return {"answer": f"{t[0]} {t[1]} {t[2]}", "triples": len(unique), "entities": entities}
        return {"answer": "No path found", "triples": 0, "entities": entities}

kg = SimpleKG()
for s, r, o in [("Netsetos","OFFERS","GenAI Course"),("Netsetos","LOCATED_IN","Hyderabad"),
    ("GenAI Course","COSTS","14999 INR"),("GenAI Course","REQUIRES","Python"),
    ("GenAI Course","REQUIRES","High School Math"),("GenAI Course","HAS_MODULE","Module 8 RAG"),
    ("Module 8 RAG","COVERS","Vector Databases"),("Module 8 RAG","COVERS","LangChain"),
    ("Module 8 RAG","COVERS","LlamaIndex"),("Module 8 RAG","COVERS","GraphRAG"),
    ("GenAI Course","HAS_POLICY","Refund Policy"),("Refund Policy","ALLOWS","Full Refund 7 Days"),
    ("Refund Policy","ALLOWS","50% Refund 8-30 Days"),("GenAI Course","INCLUDES","Discord Community"),
    ("GenAI Course","INCLUDES","GCP Credits"),("GenAI Course","HAS_MODULES","14")]:
    kg.add(s, r, o)
print(f"KG: {len(kg.triples)} triples, {len(kg.adj)} entities")

print(f"\n1-hop from GenAI Course:")
for s, r, o in kg.neighbors("genai course", 1)[:5]:
    print(f"  {s} --[{r}]--> {o}")

print(f"\n2-hop from Netsetos: {len(kg.neighbors('netsetos', 2))} triples")

print(f"\nQueries:")
for q in ["Module 8 cover?", "prerequisites GenAI?", "refund policy?", "Netsetos Hyderabad?"]:
    r = kg.query(q)
    print(f"  Q: {q} -> {r['answer'][:50]}... ({r['triples']} triples)")

print(f"\nVectors find SIMILAR. Graphs find CONNECTED.")


</details>


---
## Exercise 6 (Challenge): Hybrid Vector + Graph Router


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib
import numpy as np
from collections import defaultdict

def fe(t, dim=64):
    h = hashlib.sha256(t.lower().encode()).digest()
    v = np.array([b/255.0 for b in h[:dim]])
    for w, i in {"refund":0,"price":1,"module":2,"cover":3,"theme":4,"prereq":5,"connect":7}.items():
        if w in t.lower() and i < dim: v[i] += 0.5
    return v / np.linalg.norm(v)

chunks = ["Refund policy: full refund 7 days", "GenAI course 14999 rupees lifetime", "Prerequisites: Python and math",
          "Live classes 7 PM IST YouTube", "Module 8 covers RAG LangChain GraphRAG"]
cembs = [fe(c) for c in chunks]

def vec_rag(q):
    qe = fe(q)
    best = max(range(len(chunks)), key=lambda i: np.dot(qe, cembs[i])/(np.linalg.norm(qe)*np.linalg.norm(cembs[i])))
    return {"method": "vector", "answer": chunks[best][:45]}

gkg = defaultdict(list)
for s, r, o in [("Netsetos","OFFERS","GenAI"),("GenAI","REQUIRES","Python"),("GenAI","HAS_MODULE","Module 8"),
                ("Module 8","COVERS","GraphRAG"),("Module 8","COVERS","LangChain"),("GenAI","HAS_POLICY","Refund")]:
    gkg[s.lower()].append((s,r,o)); gkg[o.lower()].append((s,r,o))

def grp_rag(q):
    qw = set(q.lower().split())
    ts = []
    for e in gkg:
        if any(w in e for w in qw): ts.extend(gkg[e])
    if ts: return {"method": "graph", "answer": f"{ts[0][0]} {ts[0][1]} {ts[0][2]}"}
    return {"method": "graph", "answer": "No path found"}

def classify(q):
    q = q.lower()
    if any(k in q for k in ["theme","across","all","overview"]): return "thematic"
    if any(k in q for k in ["connect","relate","path","which module"]): return "relational"
    return "simple"

tests = [("Refund policy?","simple"),("Course cost?","simple"),("Live classes?","simple"),
         ("Python connected to GraphRAG?","relational"),("Which module covers LangChain?","relational"),
         ("Path from Netsetos to GraphRAG?","relational"),("Main themes across modules?","thematic"),
         ("Overview of all topics?","thematic"),("Prerequisites?","simple")]

print("Hybrid Router:")
v_ok = g_ok = h_ok = 0
for q, exp in tests:
    ct = classify(q)
    r = vec_rag(q) if ct == "simple" else grp_rag(q)
    ok = ct == exp
    h_ok += int(ok); v_ok += int(exp=="simple"); g_ok += int(exp!="simple")
    print(f"  [{'OK' if ok else '!!'}] [{ct:<11}] {q[:35]}... -> {r['answer'][:35]}...")

n = len(tests)
print(f"\nAccuracy: vector={v_ok}/{n} graph={g_ok}/{n} hybrid={h_ok}/{n}")
print(f"Hybrid wins: +15-25% accuracy, 150-200ms overhead")


</details>


---
## Exercise 7 (Challenge): GraphRAG Evaluation


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class GRE:
    @staticmethod
    def comp(ans, aspects):
        aw = set(ans.lower().split())
        return sum(1 for a in aspects if any(w in aw for w in a.lower().split())) / max(len(aspects), 1)
    @staticmethod
    def direct(ans, mx=50): return min(mx / max(len(ans.split()), 1), 1.0)

ev = GRE()
cases = [
    {"q": "Refund?", "type": "simple", "asp": ["7 days","full refund","50%"],
     "v": "Full refund 7 days 50% from 8-30 days", "g": "Refund Policy allows Full Refund 7 Days"},
    {"q": "Cost?", "type": "simple", "asp": ["14999","rupees","lifetime"],
     "v": "GenAI course 14999 rupees lifetime access", "g": "GenAI Course costs 14999 INR"},
    {"q": "Module 8 topics?", "type": "multi-hop", "asp": ["RAG","LangChain","LlamaIndex","GraphRAG"],
     "v": "Module 8 covers RAG frameworks", "g": "Module 8 RAG covers Vector Databases LangChain LlamaIndex GraphRAG"},
    {"q": "Python to Hyderabad?", "type": "multi-hop", "asp": ["Netsetos","requires","located"],
     "v": "Python prerequisite for course", "g": "Python required GenAI Course offered Netsetos located Hyderabad"},
    {"q": "Main themes?", "type": "thematic", "asp": ["AI","education","production"],
     "v": "Course covers AI topics", "g": "Themes: AI education production deployment India-focused"},
    {"q": "All offerings?", "type": "thematic", "asp": ["GenAI","community","live","credits"],
     "v": "Netsetos offers GenAI courses", "g": "Netsetos offers GenAI Course Discord live sessions GCP credits"},
]

print("GraphRAG Evaluation (4 Metrics):")
vw = gw = 0
for c in cases:
    vc = ev.comp(c["v"], c["asp"]); gc = ev.comp(c["g"], c["asp"])
    vd = ev.direct(c["v"]); gd = ev.direct(c["g"])
    vs = (vc+vd)/2; gs = (gc+gd)/2
    w = "Vector" if vs > gs else "Graph"
    if w == "Vector": vw += 1
    else: gw += 1
    print(f"  [{c['type']:<10}] {c['q']:<22} V={vs:.2f} G={gs:.2f} -> {w}")

print(f"\nVector wins: {vw}, Graph wins: {gw}")
print(f"Graph wins multi-hop+thematic, Vector wins simple")
print(f"Key: always compare both. Hybrid routing best.")


</details>


---
## Exercise 2 (Easy): Microsoft GraphRAG Quickstart
Architecture/design. See practice-lab-8_7.html.


In [ ]:
# Exercise 2: Microsoft GraphRAG Quickstart
pass


---
## Exercise 3 (Medium): Neo4j + Cypher
Architecture/design. See practice-lab-8_7.html.


In [ ]:
# Exercise 3: Neo4j + Cypher
pass


---
## Exercise 4 (Medium): LightRAG Implementation
Architecture/design. See practice-lab-8_7.html.


In [ ]:
# Exercise 4: LightRAG Implementation
pass


---
## Exercise 5 (Medium): LlamaIndex PropertyGraphIndex
Architecture/design. See practice-lab-8_7.html.


In [ ]:
# Exercise 5: LlamaIndex PropertyGraphIndex
pass


---
## Exercise 8 (Challenge): Indian Legal GraphRAG
Architecture/design. See practice-lab-8_7.html.


In [ ]:
# Exercise 8: Indian Legal GraphRAG
pass
